# Linear Attention 与混合架构

> Part 4 的长上下文一节讲了 RoPE 外推——让训练时只见过 4K 位置的模型，推理时能处理 32K、128K。这是在「保持 softmax attention 不变」的前提下，想办法让模型认识更远的位置。
>
> 这一节换一个角度：不动位置编码，改 attention 本身的复杂度。标准 softmax attention 是 $O(N^2)$，序列越长，平方增长，长上下文算不动。Linear attention、Mamba 这类方案把复杂度压到 $O(N)$。代价是表达能力下降——它们不能像 softmax 那样给某个特定 token 极高的权重。这一节从 $O(N^2)$ 瓶颈开始，理解 linear attention 怎么用结合律换复杂度，再看 MiniMax-01 等模型如何用「混合架构」兼顾两者。

标准 attention 的核心计算是 $	ext{softmax}(Q K^T) V$。一个长度为 $N$ 的序列，$Q K^T$ 是 $N 	imes N$ 的矩阵——这是 $O(N^2)$ 的来源。$N = 4096$ 时矩阵有 1600 万元素，$N = 128K$ 时直接到 170 亿元素，单层 attention 的中间结果就几十 GB。

线性注意力的核心观察：softmax 不是必须的。把它换成一个可以分离的形式 $\phi(Q) \phi(K)^T$，就能用矩阵乘法的结合律换计算顺序——先算 $\phi(K)^T V$（一个 $d 	imes d$ 的小矩阵），再乘 $\phi(Q)$。复杂度从 $O(N^2 d)$ 变成 $O(N d^2)$。当 $d \ll N$ 时（实际场景里 $d=128$、$N=32K$ 是常态），这是巨大的节省。

但 $\phi$ 不能是 softmax——softmax 的归一化让它无法分离。常用的替代是 $\phi(x) = 	ext{elu}(x) + 1$ 这类逐元素非线性。它让计算可分离，但失去了 softmax 的「尖锐性」：linear attention 给出的权重分布比较平坦，没法对某个特定 token 给极高的注意力。这就是 linear attention 在 retrieval 类任务上经常打不过 softmax 的原因。

实际生产里的解法是**混合架构**：大部分层用 linear attention（或 SSM）省计算，每隔几层插一层 softmax attention 保留「尖锐」能力。MiniMax-01 是这种思路的代表——每 8 层中 7 层 lightning attention + 1 层 softmax attention。Jamba（AI21 Labs）的混合比例不同但思路类似。

## 1. $O(N^2)$ 瓶颈：手算 attention 计算量

先把账算清楚。给定序列长度 $N$ 和 head 维度 $d$，标准 attention 的核心计算 $	ext{softmax}(Q K^T) V$ 涉及三个矩阵乘法：

```
Q × K^T  → [N, N] 矩阵，FLOPs = N × N × d = N²d
softmax  → 不算 FLOPs（按元素操作）
attn × V → [N, d] 矩阵，FLOPs = N × N × d = N²d
总 FLOPs ≈ 2 × N² × d
```

中间结果 $Q K^T$ 是个 $N 	imes N$ 矩阵，存储和计算都按 $N^2$ 增长。下面用一个具体数字感受这个增长有多快。

In [ ]:
# Softmax attention 的 O(N²) 瓶颈：手算几个具体数字

d = 128  # head_dim

print(f"head_dim = {d}")
print(f"{'序列长度 N':<15} {'QK^T 元素数':<20} {'FP16 显存':<15} {'相对 4K'}")
print("-" * 60)
base_elems = None
for N in [4096, 8192, 32768, 131072, 524288]:
    elems = N * N
    bytes_ = elems * 2  # FP16
    gb = bytes_ / (1024**3)
    if base_elems is None:
        base_elems = elems
    label = f"{N//1024}K" if N >= 1024 else str(N)
    print(f"{label:<15} {elems:<20,} {gb:<15.2f} {elems/base_elems:.0f}×")

print()
print("关键观察：从 4K 到 512K，序列长度增加 128 倍，attention 中间结果增加 16384 倍")
print("这就是为什么长上下文不能用纯 softmax attention——单层中间结果就几百 GB")

## 2. Linear Attention 的核心思想：结合律换计算顺序

线性注意力的核心观察：标准 attention 写成 $	ext{softmax}(Q K^T) V$，括号先算 $Q K^T$（得到 $N 	imes N$ 矩阵），再乘 $V$。如果把 softmax 换成一个可以分离的形式 $\phi(Q) \phi(K)^T$，括号可以换位置：

$$
\underbrace{(\phi(Q) \phi(K)^T)}_{N \times N} V \quad \Rightarrow \quad \phi(Q) \underbrace{(\phi(K)^T V)}_{d \times d}
$$

新的计算顺序：先算 $\phi(K)^T V$（一个 $d \times d$ 的小矩阵），再左乘 $\phi(Q)$。

| 计算顺序 | 中间结果大小 | 总 FLOPs |
|:---|:---|:---|
| 标准 $(Q K^T) V$ | $N \times N$ | $O(N^2 d)$ |
| Linear $Q (K^T V)$ | $d \times d$ | $O(N d^2)$ |

当 $d \ll N$ 时（比如 $d=128, N=32K$），从 $O(N^2 d)$ 到 $O(N d^2)$ 是巨大的节省。

但 softmax 不能写成 $\phi(Q) \phi(K)^T$ 的可分离形式——它对每一行做归一化，跨了所有 key。Linear attention 的做法是把 softmax 换成其他 kernel 函数，常用的有：
- $\phi(x) = \text{elu}(x) + 1$（最早由 Katharopoulos 等人 2020 提出）
- $\phi(x) = \text{ReLU}(x)$（更简单但效果稍差）
- 还有基于随机特征（random features）的近似

下面用一个 4 token 的小例子手算一遍，对比 softmax attention 和 linear attention 的输出。

In [ ]:
# 手算对比：softmax attention vs linear attention（4 token 小例子）

import torch
import torch.nn.functional as F

torch.manual_seed(42)

N = 4   # 4 个 token
d = 3   # head_dim = 3

Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

print("Q =\n", Q)
print("\nK =\n", K)
print("\nV =\n", V)

# === 标准 softmax attention ===
scores = Q @ K.T / (d ** 0.5)
attn_weights = F.softmax(scores, dim=-1)
out_softmax = attn_weights @ V
print("\n=== Softmax Attention ===")
print(f"QK^T / sqrt(d) =\n{scores}")
print(f"\nsoftmax(QK^T) =\n{attn_weights}")
print(f"\n输出 =\n{out_softmax}")

# === Linear attention ===
# φ(x) = elu(x) + 1
phi_Q = F.elu(Q) + 1
phi_K = F.elu(K) + 1

# 标准写法（O(N²d)）：先算 φ(Q)φ(K)^T，再乘 V
linear_scores_naive = phi_Q @ phi_K.T
# 归一化：每行除以该行所有分母之和
normalizer = linear_scores_naive.sum(dim=-1, keepdim=True)
out_linear_naive = (linear_scores_naive @ V) / normalizer

print("\n=== Linear Attention ===")
print(f"φ(Q) = elu(Q)+1 =\n{phi_Q}")
print(f"\nφ(K) = elu(K)+1 =\n{phi_K}")
print(f"\nφ(Q)φ(K)^T =\n{linear_scores_naive}")
print(f"\n输出 =\n{out_linear_naive}")

# === Linear attention 的 O(Nd²) 写法 ===
# 利用结合律：φ(Q) @ (φ(K)^T @ V)
KV = phi_K.T @ V  # [d, d] 小矩阵
out_linear_fast = phi_Q @ KV / normalizer
print(f"\n=== Linear Attention（结合律换顺序）===")
print(f"φ(K)^T @ V =\n{KV}")
print(f"\n输出（应该和上面相同）=\n{out_linear_fast}")
print(f"\n两次结果是否一致: {torch.allclose(out_linear_naive, out_linear_fast, atol=1e-6)}")

## 3. Linear Attention 的代价：失去「尖锐性」

Linear attention 在数学上是 softmax attention 的近似。**它们的结果不完全一样**——下面看为什么。

Softmax 的一个关键性质是**尖锐性（sharpness）**：当某个 query 和某个 key 的点积远大于其他，softmax 会把几乎所有权重给到那个 key。比如：

```
softmax([10, 1, 1, 1]) ≈ [0.999, 0.0003, 0.0003, 0.0003]  ← 极尖
```

这种「近乎 one-hot」的权重让 attention 能精准地把某个 token 的信息聚合到 query 上——这是 retrieval、copy 这类任务的关键能力。

Linear attention 没有这种尖锐性。$\phi(Q) \phi(K)^T$ 是逐元素非线性的乘积，最大值和最小值的差距远不如 softmax 大。结果是权重分布比较平坦，没法对某个特定 token 给极高权重。下面的代码量化对比两者。

In [ ]:
# Sharpness 对比：softmax vs linear attention

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 构造一个场景：query 与 key[0] 非常相似，与其他 key 不相似
torch.manual_seed(0)
N, d = 8, 16

# query 与 K[0] 几乎相同，与其他 K 正交
K = torch.randn(N, d)
Q = K[0:1].clone() + 0.01 * torch.randn(1, d)  # Q ≈ K[0]
V = torch.eye(N)  # V 是单位矩阵，这样输出就是 attention 权重本身

# Softmax attention
scores_softmax = Q @ K.T / (d ** 0.5)
weights_softmax = F.softmax(scores_softmax, dim=-1)

# Linear attention
phi_Q = F.elu(Q) + 1
phi_K = F.elu(K) + 1
linear_scores = phi_Q @ phi_K.T
weights_linear = linear_scores / linear_scores.sum(dim=-1, keepdim=True)

print(f"Query ≈ Key[0]，理想权重应该几乎全给 position 0")
print()
print(f"{'Position':<10} {'Softmax 权重':<20} {'Linear 权重':<20}")
print("-" * 50)
for i in range(N):
    print(f"{i:<10} {weights_softmax[0, i].item():<20.4f} {weights_linear[0, i].item():<20.4f}")

print()
print(f"关键观察：")
print(f"  Softmax 把 {weights_softmax[0, 0].item()*100:.1f}% 的权重给 position 0 → 近乎 one-hot")
print(f"  Linear 只把 {weights_linear[0, 0].item()*100:.1f}% 的权重给 position 0 → 分布平坦")
print(f"  → Linear attention 在「精准 retrieval」上天然弱")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(N), weights_softmax[0].numpy(), color='#e74c3c')
axes[0].set_title('Softmax Attention 权重\n（尖锐，retrieval 友好）')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Weight')
axes[0].set_ylim(0, 1)

axes[1].bar(range(N), weights_linear[0].numpy(), color='#3498db')
axes[1].set_title('Linear Attention 权重\n（平坦，retrieval 弱）')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Weight')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 4. SSM / Mamba 一瞥

Linear attention 不是唯一的 $O(N)$ 方案。另一条线是**状态空间模型**（State Space Model, SSM），从控制理论发展来，最近几年被引入到深度学习。

Mamba（2023）是 SSM 路线的代表。它的核心机制可以极简理解为：

```
SSM 的递推形式：
  h_t = A · h_{t-1} + B · x_t     # 更新隐状态
  y_t = C · h_t                    # 输出

其中 A, B, C 是学出来的参数，h_t 是隐状态向量。
```

每一步只需要做小矩阵乘法（$h$ 是 $d$ 维向量，A 是 $d \times d$ 矩阵），所以是 $O(N d^2)$，和 linear attention 一个量级。

Mamba 的关键创新是 **selective SSM**——让 $B$ 和 $C$ 依赖于输入 $x_t$，相当于「模型可以学会根据当前 token 决定要更新多少隐状态、读出多少」。这让 SSM 比 linear attention 更有表达力，可以在一些 retrieval 任务上接近 softmax attention。

Linear attention 也可以写成类似的递推形式：隐状态 $S_t = S_{t-1} + \phi(k_t) v_t^T$，输出 $o_t = \phi(q_t) S_t$。所以两者数学结构非常接近，差别在于 $\phi$ 的选择和选择性机制。下面写一个极简 SSM 演示数据流。

In [ ]:
# 极简 SSM 演示：递推形式的数据流

import torch

torch.manual_seed(42)

d = 4         # 隐状态维度
N = 6         # 序列长度
d_in = 3      # 输入维度

# SSM 参数（实际中是学出来的，这里随机初始化）
A = 0.9 * torch.eye(d)              # 状态转移矩阵（衰减）
B = torch.randn(d, d_in) * 0.1      # 输入到状态的投影
C = torch.randn(1, d)               # 状态到输出的投影

# 输入序列
x = torch.randn(N, d_in)

# 递推计算（O(N d²)）
h = torch.zeros(d)        # 初始隐状态
outputs = []
hidden_states = []

print("=== 极简 SSM 递推过程 ===")
print(f"输入序列 shape: {x.shape} (N={N}, d_in={d_in})")
print(f"隐状态维度: {d}")
print()

for t in range(N):
    # 状态更新
    h = A @ h + B @ x[t]
    hidden_states.append(h.clone())

    # 输出
    y = C @ h
    outputs.append(y.item())

    print(f"  t={t}: x={x[t].tolist()}, h[:2]={h[:2].tolist()}, y={y.item():.4f}")

print()
print(f"关键观察：每步只需 {d*d + d*d_in + d} 次乘加（O(d²)），与序列长度无关")
print(f"隐状态固定 {d} 维——不像 attention 那样要存所有历史")
print(f"这是 SSM / linear attention 在长序列上高效的根本原因")

## 5. Lightning Attention：MiniMax-01 的工程创新

Linear attention 的「$O(N d^2)$ 复杂度」是数学上的，实际工程上有一个麻烦：递推形式（一步算一个 token）很难在 GPU 上高效跑。GPU 的强项是大矩阵乘法，逐步递推会让算力利用率掉到很低。

MiniMax-01 的 **Lightning Attention** 解决了这个问题。核心思路是利用**因果掩码下的特殊结构**——前面 token 不能看后面，但**左半部分（无因果掩码）可以一次算完**，右半部分（需要因果掩码）才用递推。这样大部分计算走大矩阵乘法（GPU 友好），只有一小部分走递推。

具体细节较深，本节只介绍到「Lightning Attention 让 linear attention 的训练速度和 softmax attention 接近」这一点。MiniMax-01 论文报告：相比朴素 linear attention，Lightning Attention 训练速度提升约 11 倍，达到 softmax attention 同等量级。

工程上的另一个贡献是：把 lightning attention 和 softmax attention 在同一个模型里混合使用，下一节展开。

## 6. Hybrid 架构：linear + softmax 混合

第 3 节看到 linear attention 在 retrieval 类任务上天然弱。但生产场景里，长上下文 + 多轮对话 + 工具调用都需要精准 retrieval——纯 linear attention 模型在这些任务上会显著掉点。

工程上的解法是**混合架构**：大部分层用 linear attention / SSM（省计算），每隔几层插一层 softmax attention（保留尖锐能力）。

| 模型 | 总层 | Linear/SSM 层 | Softmax 层 | 比例 |
|:---|:---|:---|:---|:---|
| **MiniMax-01** | 80 | 70 (lightning attn) | 10 (softmax attn) | 7:1 |
| **Jamba** | 32 | 24 (Mamba) | 8 (softmax attn) + MoE | 3:1（局部） |
| **Zamba** | 多 | 大部分 Mamba | 少量共享 softmax | 不同设计 |

为什么这个比例有效：linear attention 处理「平滑、积累型」的信息流（如长上下文理解、风格保持），softmax attention 处理「尖锐、定位型」的 retrieval。混合后两种能力都有了。

下面用代码模拟一个 hybrid block 序列，直观感受层类型分布。

In [ ]:
# 模拟 MiniMax-01 的 hybrid 架构：每 8 层中 7 层 lightning + 1 层 softmax

import matplotlib.pyplot as plt
import numpy as np

num_layers = 80
layer_types = []
for i in range(num_layers):
    # MiniMax-01 的设计：每 8 层里最后 1 层是 softmax
    if (i + 1) % 8 == 0:
        layer_types.append('softmax')
    else:
        layer_types.append('lightning')

# 可视化
fig, ax = plt.subplots(figsize=(14, 3))

colors = []
for lt in layer_types:
    if lt == 'softmax':
        colors.append('#e74c3c')
    else:
        colors.append('#3498db')

ax.bar(range(num_layers), [1]*num_layers, color=colors, edgecolor='black', linewidth=0.3)
ax.set_xlabel('Layer index')
ax.set_ylabel('Type')
ax.set_title(f'MiniMax-01 风格 hybrid 架构：{num_layers} 层，红色=softmax attention，蓝色=lightning attention')
ax.set_yticks([])

# 标注前 16 层
for i in range(min(16, num_layers)):
    label = 'S' if layer_types[i] == 'softmax' else 'L'
    ax.text(i, 0.5, label, ha='center', va='center', color='white', fontsize=9, fontweight='bold')

ax.set_xlim(-0.5, num_layers - 0.5)
plt.tight_layout()
plt.show()

# 统计
n_softmax = sum(1 for lt in layer_types if lt == 'softmax')
n_lightning = num_layers - n_softmax
print(f"总层: {num_layers}")
print(f"Softmax 层: {n_softmax} ({n_softmax/num_layers*100:.0f}%)")
print(f"Lightning 层: {n_lightning} ({n_lightning/num_layers*100:.0f}%)")
print()
print("关键观察：10% 的 softmax attention 层提供了 retrieval 能力")
print("剩下 90% 的 lightning attention 层负责长序列的高效处理")

## 7. 实测对比：复杂度差异

把三种 attention（softmax / linear / linear-recurrent）放在不同序列长度下计时，看复杂度的实际影响。

注意：教学代码没有做 kernel 优化，绝对数字不代表生产性能，但**相对增长趋势**能反映复杂度差异。

In [ ]:
# 复杂度对比：softmax vs linear attention（递推形式） vs linear（向量化形式）

import torch
import torch.nn.functional as F
import time

def softmax_attention(Q, K, V):
    """标准 softmax attention, O(N²d)"""
    scores = Q @ K.T / (Q.shape[-1] ** 0.5)
    weights = F.softmax(scores, dim=-1)
    return weights @ V

def linear_attention_vectorized(Q, K, V):
    """Linear attention，向量化形式（用结合律），O(Nd²)"""
    phi_Q = F.elu(Q) + 1
    phi_K = F.elu(K) + 1
    # 利用结合律：先算 KV（d×d），再算 Q @ KV
    KV = phi_K.T @ V
    out = phi_Q @ KV
    # 归一化：normalizer[i] = sum_j phi_Q[i] · phi_K[j] = phi_Q[i] · sum_j phi_K[j]
    normalizer = (phi_Q @ phi_K.sum(dim=0)).unsqueeze(-1).clamp(min=1e-6)
    return out / normalizer

def linear_attention_recurrent(Q, K, V):
    """Linear attention，递推形式（教学，慢），O(Nd²) 但常数大"""
    N, d = Q.shape
    phi_Q = F.elu(Q) + 1
    phi_K = F.elu(K) + 1
    S = torch.zeros(d, d)  # 累积状态
    outputs = []
    for t in range(N):
        # 更新状态
        S += phi_K[t].unsqueeze(1) @ V[t].unsqueeze(0)
        # 输出
        out = phi_Q[t] @ S
        outputs.append(out)
    return torch.stack(outputs)

# 在不同序列长度下计时
d = 64
seq_lens = [256, 512, 1024, 2048, 4096]
results = {'softmax': [], 'linear_vec': [], 'linear_rec': []}

for N in seq_lens:
    Q = torch.randn(N, d)
    K = torch.randn(N, d)
    V = torch.randn(N, d)

    for name, fn in [('softmax', softmax_attention),
                     ('linear_vec', linear_attention_vectorized),
                     ('linear_rec', linear_attention_recurrent)]:
        # warmup
        if N <= 1024 or name != 'linear_rec':
            fn(Q, K, V)

        # 计时
        tries = []
        for _ in range(3):
            t0 = time.time()
            for _ in range(5):
                fn(Q, K, V)
            tries.append((time.time() - t0) / 5)
        results[name].append(min(tries))

# 可视化
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(seq_lens, results['softmax'], 'o-', label='Softmax O(N²d)', linewidth=2, color='#e74c3c')
plt.plot(seq_lens, results['linear_vec'], 's-', label='Linear (vectorized) O(Nd²)', linewidth=2, color='#3498db')
plt.plot(seq_lens, results['linear_rec'], '^-', label='Linear (recurrent)', linewidth=2, color='#f39c12')
plt.xlabel('Sequence length N')
plt.ylabel('Time per forward (s)')
plt.title('Attention 复杂度对比（教学代码，趋势代表复杂度差异）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log', base=2)
plt.yscale('log')
plt.xticks(seq_lens, [str(s) for s in seq_lens])
plt.tight_layout()
plt.show()

print("关键观察：")
print("  1. Softmax attention 时间随 N² 增长，最长序列上明显比 linear 慢")
print("  2. Vectorized linear attention 几乎线性增长（实际是 Nd²，d 固定时近似线性）")
print("  3. Recurrent 形式虽然复杂度也是 O(Nd²)，但常数大、GPU 利用率低 → 实际更慢")
print("     这就是为什么 Lightning Attention 工程上很重要")

## 小结

- [ ] 标准 softmax attention 是 $O(N^2 d)$，序列长度增加 128 倍时中间结果增加 16384 倍
- [ ] Linear attention 用 $\phi(Q)\phi(K)^T$ 替代 softmax，可通过结合律换计算顺序，复杂度降到 $O(N d^2)$
- [ ] Linear attention 失去 softmax 的尖锐性，在 retrieval 类任务上天然弱
- [ ] SSM/Mamba 是另一种 $O(N)$ 路线，selective SSM 让 B、C 依赖输入，比 linear attention 更有表达力
- [ ] Linear attention 的递推形式在 GPU 上效率低，Lightning Attention（MiniMax-01）用左半部分向量化+右半部分递推的混合策略让训练速度接近 softmax attention
- [ ] Hybrid 架构：大部分层用 linear/SSM，每隔几层插 softmax，兼顾效率和 retrieval 能力
- [ ] MiniMax-01 是 7:1（每 8 层 7 lightning + 1 softmax），Jamba、Zamba 类似思路但比例不同

参考：[Linear Transformer](https://arxiv.org/abs/2006.16236)、[Mamba](https://arxiv.org/abs/2312.00752)、[MiniMax-01](https://arxiv.org/abs/2501.08313)、[Jamba](https://arxiv.org/abs/2403.19887)。

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：复杂度计算**

给定 $N = 32768$（32K 上下文），$d = 128$（head_dim）。算标准 softmax attention 和 linear attention 的总 FLOPs，以及它们的比值。

小提示：softmax attention 总 FLOPs $\approx 2 N^2 d$（QK^T 和 attn×V 各一份）。Linear attention 总 FLOPs $\approx 2 N d^2$（KV 和 Q×KV 各一份）。

In [ ]:
# 作业 1：复杂度计算

N = 32768
d = 128

# TODO: 填入计算
softmax_flops = None   # 2 * N * N * d
linear_flops = None    # 2 * N * d * d
ratio = None           # softmax_flops / linear_flops，越大说明 linear 越省

assert softmax_flops is not None, "请先计算 softmax_flops"
assert linear_flops is not None, "请先计算 linear_flops"
assert ratio is not None, "请先计算 ratio"

expected_softmax = 2 * N * N * d
expected_linear = 2 * N * d * d
expected_ratio = expected_softmax / expected_linear

assert softmax_flops == expected_softmax
assert linear_flops == expected_linear
assert abs(ratio - expected_ratio) < 0.001

print(f"✅ 作业 1 通过")
print(f"   Softmax attention FLOPs: {softmax_flops:,} ({softmax_flops:.2e})")
print(f"   Linear attention FLOPs:  {linear_flops:,} ({linear_flops:.2e})")
print(f"   比值: {ratio:.0f}× → Linear attention 省 {ratio:.0f} 倍计算")
print(f"   关键观察：N >> d 时，linear attention 的节省非常显著")

**作业 2：实现 linear attention 的向量化形式**

下面函数省略了核心计算。补全：用 $\phi(x) = \text{elu}(x) + 1$，通过结合律（先算 $\phi(K)^T V$ 再乘 $\phi(Q)$）实现 linear attention 的向量化形式。

小提示：`phi_Q = F.elu(Q) + 1`，`phi_K = F.elu(K) + 1`，`KV = phi_K.T @ V`，`out = phi_Q @ KV`。归一化用每行的总和除。

In [ ]:
# 作业 2：实现 linear attention 的向量化形式

import torch
import torch.nn.functional as F

def linear_attention_vectorized(Q, K, V):
    """Linear attention 的向量化形式（O(Nd²)）

    使用结合律：先算 φ(K)^T @ V（d×d 矩阵），再算 φ(Q) @ 该矩阵
    φ(x) = elu(x) + 1
    """
    # TODO: 补全下面几行
    phi_Q = None
    phi_K = None
    KV = None       # phi_K.T @ V
    out = None      # phi_Q @ KV
    # 归一化（每行除以该行的总分）
    normalizer = None  # (phi_Q @ phi_K.sum(dim=0)).clamp(min=1e-6)
    return out / normalizer

# 验证
torch.manual_seed(42)
N, d = 16, 8
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

out = linear_attention_vectorized(Q, K, V)

# 用朴素 O(N²d) 写法对比
def linear_attention_naive(Q, K, V):
    phi_Q = F.elu(Q) + 1
    phi_K = F.elu(K) + 1
    scores = phi_Q @ phi_K.T
    return (scores @ V) / scores.sum(dim=-1, keepdim=True)

out_ref = linear_attention_naive(Q, K, V)
assert out.shape == (N, d), f"输出 shape 应为 ({N}, {d})，实际 {out.shape}"
assert torch.allclose(out, out_ref, atol=1e-5), "和朴素写法结果不一致"

print(f"✅ 作业 2 通过")
print(f"   输出 shape: {out.shape}")
print(f"   和朴素写法一致: {torch.allclose(out, out_ref, atol=1e-5)}")

**作业 3：Hybrid 架构的层类型分布**

给定一个 hybrid 模型共 32 层，每 4 层中第 4 层是 softmax attention（其余是 linear）。算出 softmax 层占比。

小提示：循环 32 层，判断 `(i+1) % 4 == 0` 来标记 softmax 层。

In [ ]:
# 作业 3：Hybrid 架构的层类型分布

num_layers = 32
period = 4  # 每 4 层中第 4 层是 softmax

# TODO: 补全计算
layer_types = None   # list of 'softmax' or 'linear'，长度为 num_layers
n_softmax = None     # softmax attention 层数
softmax_ratio = None # n_softmax / num_layers

assert layer_types is not None
assert n_softmax is not None
assert softmax_ratio is not None

expected_types = ['softmax' if (i+1) % period == 0 else 'linear' for i in range(num_layers)]
expected_n = sum(1 for t in expected_types if t == 'softmax')
expected_ratio = expected_n / num_layers

assert layer_types == expected_types
assert n_softmax == expected_n
assert abs(softmax_ratio - expected_ratio) < 0.001

print(f"✅ 作业 3 通过")
print(f"   总层: {num_layers}")
print(f"   Softmax 层: {n_softmax} ({softmax_ratio*100:.0f}%)")
print(f"   Linear 层: {num_layers - n_softmax} ({(1-softmax_ratio)*100:.0f}%)")
print(f"   关键观察：即使 25% 的层是 softmax，retrieval 能力也通常足够")